# 1. OSA GROWTH INTELLIGENCE MODEL
**Objective:** Predict **Next Week's Netflow Rate** ($NET_{t+1}$) for strategic pricing and liquidity planning.

### **The Pipeline Architecture**
This notebook serves as a weekly operational pipeline designed to run every **Friday morning**. Here is the mechanics of the flow:

1. **Temporal Matrix (Forward Calendar vs Lagged Macro):** 
   * **Forward-Looking (t+1):** We absolutely know the upcoming week's calendar (e.g., scheduled PPK meetings or strict YearEnd dates). Therefore, calendar flags for the $t+1$ prediction are extracted directly from the $t+1$ schedule.
   * **Historical (t):** Macro indicators (TLREF) and customer inertia (NET_lag1) are strictly localized to the *completed week* $t$ because they act as the historical springboard driving next week's results.
2. **Dynamic Learning (Sliding Window):** The pipeline utilizes a continuous sliding-window validation over the last 52 weeks. By dynamically discarding data older than 52 weeks, the regression engine unlearns obsolete inflation/interest regimes, remaining highly agile.
3. **Execution Ready (Production):** The final cell acts as the live engine. Upon running the notebook, you declare the upcoming calendar flags, and the model outputs a 95% Confidence Interval for the immediate upcoming week.

### **Methodology**
* **Model A (Macro Baseline):** A purely exogenous model forecasting liquidity based on macroeconomic indicators (`w/TLREF`) and forward calendar triggers (`Target_PPK`, `Target_YearEnd`).
* **Model B (Macro + Momentum):** An enhanced structure combining macro features with *Internal Autoregressive* signals (`NET_lag1`). It evaluates actual historical customer behavioral inertia alongside theoretical interest rates.

In [ ]:
# Import standard data manipulation and visualization suites
import pandas as pd
import numpy as np

# Import econometric and machine learning libraries
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error, mean_squared_error, roc_auc_score
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Import visualization tools
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from IPython.display import display, Markdown

import os, warnings
warnings.filterwarnings('ignore')

# aesthetic configurations for corporate reporting
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)

# Define strict color schema for charts
BLUE = '#1f77b4'
ORANGE = '#ff7f0e'
GRAY_DARK = '#333333'
print('Libraries imported and environment initialized.')

# 2. DATA INGESTION & TEMPORAL MATRIX
Here we build the operational shift mapping.
* $Target$ is shifted backwards (`.shift(-1)`) bringing the upcoming week's flow into the current completed week.
* $Target\_PPK$ and $Target\_YearEnd$ are also shifted backwards (`.shift(-1)`) since schedule dates are publicly guaranteed in advance! 
* All other metrics (TLREF, Momentum) remain unshifted because they strictly summarize the week that just occurred.

In [ ]:
# 1. Load the unified dataset compiled from the upstream SQL layer.
df_raw = pd.read_excel('FORA_MODEL_MERGED.xlsx')
df_raw['Date'] = pd.to_datetime(df_raw['max_tarih'], format='%d.%m.%Y')
df_raw = df_raw.rename(columns={'netflow_rate': 'NET'})

# 2. Chronological sequencing is critical for time-series validity.
df = df_raw.sort_values('Date').reset_index(drop=True)

# 3. TEMPORAL SHIFT (The Core Mechanic)
# We define 'Target' as the mathematically next week's netflow (t+1).
df['Target'] = df['NET'].shift(-1)

# Calendar flags for the TARGET week (t+1). Known with 100% certainty in advance.
df['Target_PPK'] = df['PPK'].shift(-1)
df['Target_YearEnd'] = df['YearEnd'].shift(-1)

# 4. Feature Engineering
# The realized netflow of the completed week (t) acts as the Momentum for the Target.
df['NET_lag1'] = df['NET']
# We calculate a 3-week rolling average of the recent flows to capture prolonged inertia.
df['NET_roll3_lag1'] = df['NET'].rolling(window=3).mean()

# 5. Define Feature Spaces
base_features = ['w/TLREF', 'Target_PPK', 'Target_YearEnd', 'EXP(CBavg-TLREF)', 'Market_Anomaly']
dynamic_features = base_features + ['NET_lag1', 'NET_roll3_lag1']

# 6. Sanitize NaN arrays caused by `.shift(-1)` at the end of the dataframe, 
# and `.rolling(3)` at the beginning of the dataframe.
df = df.dropna(subset=['Target'] + dynamic_features).reset_index(drop=True)

print(f'Total Usable Historical Rows for Training: {len(df)}')
display(df[['Date', 'NET', 'w/TLREF', 'Target_PPK', 'Target_YearEnd', 'Target']].head())

# 3. ANALYTICAL FRAMEWORK
This section builds the dynamic loop. Instead of splitting data simply as "train" and "test", we roll through time. At every step `t`, the engine trains an OLS formulation using strictly the `t-52` history, evaluates exactly one step forward (`t`), records the out-of-sample error, and moves forward by a week.

In [ ]:
def walk_forward_ols(df_input, features, window_size=52, model_name='Model'):
    """
    Executes a 52-Week Sliding Window Walk-Forward Validation.
    """
    dfc = df_input.copy()
    n = len(dfc)
    
    start = min(window_size, n - 1)

    preds, acts, dts = [], [], []
    r2s, sizes = [], []
    coef_hist = []
    last_mdl = None

    for t in range(start, n):
        # SLIDING WINDOW: Drops data older than 52 weeks to adapt to regime shifts.
        ts = max(0, t - window_size) 
        
        # Train strictly on the [ts to t-1] history
        tr = dfc.iloc[ts:t]
        
        # Test out-of-sample purely on step [t]
        te = dfc.iloc[t:t+1]

        Xr = tr[features].values.astype(np.float64)
        yr = tr['Target'].values.astype(np.float64)
        Xe = te[features].values.astype(np.float64)

        if np.isnan(Xr).any() or np.isnan(yr).any() or np.isnan(Xe).any(): continue

        Xr_c = np.column_stack([np.ones(len(Xr)), Xr])
        Xe_c = np.column_stack([np.ones(len(Xe)), Xe])
        
        mdl = sm.OLS(yr, Xr_c).fit()
        p = mdl.predict(Xe_c)[0]
        
        preds.append(p)
        acts.append(te['Target'].values[0])
        dts.append(te['Date'].values[0])
        sizes.append(len(tr))
        r2s.append(mdl.rsquared)
        
        cd = {'Date': te['Date'].values[0], 'const': mdl.params[0]}
        for j, f in enumerate(features): cd[f] = mdl.params[j+1]
        coef_hist.append(cd)
        
        last_mdl = mdl

    res = pd.DataFrame({'Date': dts, 'Actual': acts, 'Predicted': preds})
    res['Error'] = res['Actual'] - res['Predicted']

    mae = mean_absolute_error(res['Actual'], res['Predicted'])
    rmse = np.sqrt(mean_squared_error(res['Actual'], res['Predicted']))
    da = (np.sign(res['Actual']) == np.sign(res['Predicted'])).mean()

    met = {
        'Model': model_name, 'Window': f'{window_size}w', 'MAE': mae, 'RMSE': rmse,
        'Direction Acc': da, 'Avg Train R2': np.mean(r2s), 'N': len(res)
    }
    return res, last_mdl, met, pd.DataFrame(coef_hist)

def plot_forecast(res, title, color):
    """ Renders comparing actual trajectories versus OLS predicted pipelines. """
    fig, ax = plt.subplots(figsize=(16, 6))
    ax.plot(res['Date'], res['Actual'], label='Actual NET', color=GRAY_DARK, alpha=0.4, linewidth=3)
    ax.plot(res['Date'], res['Predicted'], label=f'Predicted ({title})', color=color, linewidth=2, linestyle='--', marker='s', markersize=4)
    ax.axhline(0, color='black', linewidth=0.5, linestyle=':')
    ax.set_title(f'{title} Out-of-Sample Validation', fontsize=15, fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.2)
    plt.show()

def get_gauc_metrics(res, pred_col='Predicted', target_col='Actual'):
    try:
        med = res[target_col].median()
        y_true = (res[target_col] >= med).astype(int)
        auc = roc_auc_score(y_true, res[pred_col])
        status = "GREEN" if auc > 0.65 else ("YELLOW" if auc >= 0.60 else "RED")
        return auc, status
    except:
        return np.nan, "N/A"

def print_detailed_stats(model, model_name, res_df):
    """ Spits corporate tables regarding P-Values, Cond-No, VIF diagnostics. """
    print(f"\n{'-'*20} STATISTICAL DIAGNOSTICS: {model_name} {'-'*20}")
    
    cond_no = model.condition_number
    test_mae = mean_absolute_error(res_df['Actual'], res_df['Predicted'])
    test_rmse = np.sqrt(mean_squared_error(res_df['Actual'], res_df['Predicted']))

    metrics = pd.DataFrame({
        'Metric': ['Train R-Squared', 'Train Adj. R-Squared', 'Test MAE', 'Test RMSE', 'Condition Number'],
        'Value': [f'{model.rsquared:.4f}', f'{model.rsquared_adj:.4f}', f'{test_mae:.4f}', f'{test_rmse:.4f}', f'{cond_no:,.0f}']
    })
    display(metrics)

    try: X = model.model.exog
    except: X = None
    
    coef_data = []
    try: idx_list = model.params.index
    except: idx_list = range(len(model.params))
    
    for i, idx in enumerate(idx_list):
        p_val = model.pvalues[idx] if hasattr(model.pvalues, 'index') else model.pvalues[i]
        sig = "***" if p_val < 0.01 else ("**" if p_val < 0.05 else ("*" if p_val < 0.1 else ""))
        vif = np.nan
        if X is not None:
            try: vif = variance_inflation_factor(X, i)
            except: pass
            
        coef_data.append({
            'Variable': idx, 
            'Coef': model.params[idx] if hasattr(model.params, 'index') else model.params[i], 
            'P-Value': p_val, 
            'VIF': vif,
            'Sig': sig
        })
    display(pd.DataFrame(coef_data))
    
    gauc, status = get_gauc_metrics(res_df)
    print(f"\n[G-AUC Metric]: {gauc:.4f} ({status})")
    print("="*80)


# 4. MODEL A (Baseline Macro)
Isolating the prediction strictly on external variables (`w/TLREF`, `Target_PPK`, `Target_YearEnd`) without giving the model any knowledge regarding historical flows.

In [ ]:
res_A, mdl_A, met_A, coef_A = walk_forward_ols(
    df, base_features, window_size=52, model_name='Model A (Baseline)'
)

display(pd.DataFrame([met_A]))
plot_forecast(res_A, 'Model A', ORANGE)

print('\n--- LAST 10 WEEKS: ACTUAL VS PREDICTED (Model A) ---')
res_disp_A = res_A[['Date', 'Actual', 'Predicted', 'Error']].copy()
res_disp_A['Date'] = res_disp_A['Date'].dt.strftime('%Y-%m-%d')
display(res_disp_A.tail(10).round(3))

print_detailed_stats(mdl_A, 'Model A', res_A)

# 5. MODEL B (Macro + Momentum)
Injects Autoregressive features (`NET_lag1` and `NET_roll3_lag1`) to observe the difference statistical learning yields when client inertia is analyzed alongside macro rates.

In [ ]:
res_B, mdl_B, met_B, coef_B = walk_forward_ols(
    df, dynamic_features, window_size=52, model_name='Model B (Enhanced)'
)

display(pd.DataFrame([met_B]))
plot_forecast(res_B, 'Model B', BLUE)

print('\n--- LAST 10 WEEKS: ACTUAL VS PREDICTED (Model B) ---')
res_disp_B = res_B[['Date', 'Actual', 'Predicted', 'Error']].copy()
res_disp_B['Date'] = res_disp_B['Date'].dt.strftime('%Y-%m-%d')
display(res_disp_B.tail(10).round(3))

print_detailed_stats(mdl_B, 'Model B', res_B)

# 6. PERFORMANCE SUMMARY
Juxtapose models mathematically.

In [ ]:
print('='*80)
print('EVALUATION MATRIX: Model A vs Model B')
print('='*80)
comp_df = pd.DataFrame([met_A, met_B])
display(comp_df)

# 7. PRODUCTION FORECAST (LIVE)
Generates the out-of-sample mathematical prediction for the upcoming week based on known guarantees (Calendar) and finalized history (Macro/Momentum).

In [ ]:
# --- USER INPUT (UPCOMING WEEK) ---
NEXT_WEEK_PPK = 0       # Is there a PPK meeting next week? (1 or 0)
NEXT_WEEK_YEAREND = 0   # Is next week considered YearEnd? (1 or 0)
# ----------------------------------

# 1. Dynamically target the endpoint of the dataset.
last_date = df_raw['Date'].iloc[-1]
next_date = last_date + pd.Timedelta(days=7)

# 2. Construct the prediction matrix for the future
prod_df = pd.DataFrame([{
    'w/TLREF': df_raw['w/TLREF'].iloc[-1],                     # Completed week closing rate
    'Target_PPK': NEXT_WEEK_PPK,                               # Known upcoming week flag
    'Target_YearEnd': NEXT_WEEK_YEAREND,                       # Known upcoming week flag
    'EXP(CBavg-TLREF)': df_raw['EXP(CBavg-TLREF)'].iloc[-1],   # Completed week condition
    'Market_Anomaly': df_raw['Market_Anomaly'].iloc[-1],       # Completed week condition
    'NET_lag1': df_raw['NET'].iloc[-1],                        # Completed week momentum
    'NET_roll3_lag1': df_raw['NET'].iloc[-3:].mean()           # Completed week trend
}])

print('='*60)
print('     PRODUCTION ENGINE: UPCOMING NETFLOW T+1')
print('='*60)
print(f"Target Week Boundary : {next_date.strftime('%Y-%m-%d')}")
print(f"Calendar Event (PPK) : {prod_df['Target_PPK'].values[0]}")
print(f"Historical Flow (NET): {prod_df['NET_lag1'].values[0]:+.2f}")
print()

# 3. Process array through Model B
X_prod = np.column_stack([np.ones(1), prod_df[dynamic_features].values.astype(np.float64)])
pred_final = mdl_B.predict(X_prod)[0]
ci = mdl_B.get_prediction(X_prod).summary_frame(alpha=0.05)

print(f">>> FINAL FORECAST (Model B) : {pred_final:+.2f}")
print(f">>> 95% CONFIDENCE INTERVAL  : [{ci['obs_ci_lower'].values[0]:+.2f}, {ci['obs_ci_upper'].values[0]:+.2f}]")
print('='*60)
